# ChurnSense AI — EDA walkthrough

A narrated tour of the analysis. **Every computation here calls the
`churnsense` package**; nothing is reimplemented in a cell. That is deliberate:
a notebook that holds its own copy of "churn rate by contract" will eventually
disagree with the report and the dashboard.

For the full written analysis see [`reports/eda_report.md`](../reports/eda_report.md),
regenerated by `make eda`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd

from churnsense import analysis as an
from churnsense.data import schema
from churnsense.data.loader import load_clean

pd.set_option("display.width", 110)

## 1. Load and clean

The loader reports every change it makes.

In [2]:
frame, cleaning = load_clean()
print(f"{len(frame):,} customers x {frame.shape[1]} columns\n")
for line in cleaning.as_lines():
    print(" -", line)

02:32:11 INFO     churnsense.data.loader       read raw rows=7043 cols=21 path=Telco-Customer-Churn.csv


02:32:11 INFO     churnsense.data.loader       clean: rows: 7043 in -> 7043 out


02:32:11 INFO     churnsense.data.loader       clean: exact duplicate rows dropped: 0


02:32:11 INFO     churnsense.data.loader       clean: blank TotalCharges found: 11 (filled with 0.0 where tenure == 0: 11; filled from tenure x MonthlyCharges: 0)


02:32:11 INFO     churnsense.data.loader       clean: whitespace stripped in: TotalCharges


7,043 customers x 21 columns

 - rows: 7043 in -> 7043 out
 - exact duplicate rows dropped: 0
 - blank TotalCharges found: 11 (filled with 0.0 where tenure == 0: 11; filled from tenure x MonthlyCharges: 0)
 - whitespace stripped in: TotalCharges


The 11 blank `TotalCharges` values all sit at `tenure == 0` — customers who have
not been billed yet — so they are filled with `0.00`. Median imputation would
have invented roughly $1,400 of spend for a brand-new customer, precisely in the
tenure region the model is most sensitive to.

In [3]:
churned = frame[schema.TARGET].eq(1)
print(f"churn rate            {churned.mean():.4f}")
print(f"churned / retained    {churned.sum():,} / {(~churned).sum():,}")
print(f"imbalance ratio       1 : {(~churned).sum() / churned.sum():.1f}")
print(f"accuracy of 'nobody churns'  {(~churned).mean():.1%}  <- why accuracy is not the headline")

churn rate            0.2654
churned / retained    1,869 / 5,174
imbalance ratio       1 : 2.8
accuracy of 'nobody churns'  73.5%  <- why accuracy is not the headline


## 2. Where churn concentrates

The denominator travels with every rate: a 100% rate over three customers is noise.

In [4]:
an.churn_rate_by(frame, "Contract", sort_by_rate=True)

,Contract,customers,churned,churn_rate
0,Month-to-month,3875,1655,0.427097
1,One year,1473,166,0.112695
2,Two year,1695,48,0.028319


In [5]:
an.churn_rate_by(frame.assign(tenure_bucket=an.tenure_bucket(frame["tenure"])), "tenure_bucket")

,tenure_bucket,customers,churned,churn_rate
0,0-3,1062,597,0.562147
1,4-6,419,187,0.446301
2,7-12,705,253,0.358865
3,13-24,1024,294,0.287109
4,25-48,1594,325,0.203890
5,49-72,2239,213,0.095132


Risk is front-loaded. Note the self-selection confound: customers who expect to
stay are also the ones willing to sign a two-year contract, so part of that
15x spread is *who chooses what*, not an effect of the contract itself. The
snapshot cannot separate the two.

## 3. Price is not monotonic

The finding that argues for a non-linear model.

In [6]:
an.churn_rate_by(frame.assign(charge_band=an.charge_band(frame["MonthlyCharges"])), "charge_band")

,charge_band,customers,churned,churn_rate
0,<$35,1731,188,0.108608
1,$35-55,889,246,0.276715
2,$55-75,1295,351,0.271042
3,$75-95,1820,662,0.363736
4,$95+,1308,422,0.322630


Churn **peaks** in the $75-95 band and **falls** above $95. The top band is
dominated by long-tenure customers holding several services, so price and
commitment pull against each other there. "Higher price drives churn" does not
fit this shape.

## 4. Ranked association — and the two measures in it

In [7]:
ranking = an.association_ranking(frame.drop(columns=[schema.ID_COLUMN]))
ranking.head(10)

,feature,measure,strength,direction
0,Contract,Cramer's V,0.409798,n/a (categorical)
1,tenure,point-biserial |r|,0.352229,higher -> less churn
2,OnlineSecurity,Cramer's V,0.347016,n/a (categorical)
3,TechSupport,Cramer's V,0.342526,n/a (categorical)
4,InternetService,Cramer's V,0.322037,n/a (categorical)
5,PaymentMethod,Cramer's V,0.302677,n/a (categorical)
6,OnlineBackup,Cramer's V,0.291850,n/a (categorical)
7,DeviceProtection,Cramer's V,0.281095,n/a (categorical)
8,StreamingMovies,Cramer's V,0.230351,n/a (categorical)
9,StreamingTV,Cramer's V,0.229902,n/a (categorical)


In [8]:
gender = ranking.loc[ranking["feature"] == "gender", "strength"].iloc[0]
print(f"gender association strength: {gender:.4f}")
print("-> no detectable signal, and a protected attribute: excluded from the model")

gender association strength: 0.0000
-> no detectable signal, and a protected attribute: excluded from the model


Two statistics share one column: bias-corrected Cramer's V for categoricals and
|point-biserial r| for numerics. Both are bounded in [0, 1] and vanish under
independence, so the ranking is meaningful — but they are not the same quantity,
which is why `measure` is reported alongside.

**This ranking is not the SHAP ranking.** Contract is first here and fourth by
SHAP, because it overlaps heavily with tenure and its *unique* multivariate
contribution is smaller. Both are published rather than reconciled.

## 5. Collinearity, not leakage

In [9]:
import numpy as np

r = np.corrcoef(frame["tenure"] * frame["MonthlyCharges"], frame["TotalCharges"])[0, 1]
print(f"corr(tenure x MonthlyCharges, TotalCharges) = {r:.4f}")
print("All three are known at prediction time, so this is redundancy, not leakage.")
print("Kept, with the consequence (unstable linear coefficients) documented.")

corr(tenure x MonthlyCharges, TotalCharges) = 0.9996
All three are known at prediction time, so this is redundancy, not leakage.
Kept, with the consequence (unstable linear coefficients) documented.


## 6. Add-ons, restricted to customers who can buy them

In [10]:
an.service_addon_rates(frame)

,addon,with_addon,without_addon,n_with,n_without,difference
0,Online security add-on,0.146112,0.417667,2019,3498,-0.271555
1,Tech support add-on,0.151663,0.416355,2044,3473,-0.264691
2,Online backup add-on,0.215315,0.399288,2429,3088,-0.183973
3,Device protection add-on,0.225021,0.391276,2422,3095,-0.166256
4,Streaming movies,0.299414,0.336804,2732,2785,-0.037390
5,Streaming TV,0.300702,0.335231,2707,2810,-0.034529


Internet subscribers only — for everyone else these columns read
"No internet service" and the comparison would be meaningless.

Support-shaped add-ons show the widest gaps. This is **not** evidence that
giving someone tech support retains them: customers who buy support may simply
be more invested already.

## 7. Two-way concentration

Cells under 20 customers are suppressed rather than reported.

In [11]:
rate, counts = an.risk_concentration(frame)
display(rate.style.format("{:.1%}", na_rep="n<20").set_caption("Churn rate"))
counts

tenure_bucket,0-3,4-6,7-12,13-24,25-48,49-72
Contract,,,,,,
Month-to-month,58.0%,47.7%,42.0%,37.7%,32.9%,26.0%
One year,9.5%,n<20,10.6%,8.1%,10.6%,12.9%
Two year,n<20,n<20,0.0%,0.0%,2.2%,3.3%


tenure_bucket,0-3,4-6,7-12,13-24,25-48,49-72
Contract,,,,,,
Month-to-month,1025,388,581,737,802,342
One year,21,18,85,197,518,634
Two year,16,13,39,90,274,1263


---

## Where to go next

| | |
|---|---|
| Full EDA with figures | `make eda` -> `reports/eda_report.md` |
| Model comparison | `make train` -> `reports/model_comparison.md` |
| Test-set evaluation | `make evaluate` -> `reports/final_evaluation.md` |
| Interactive exploration | `make app` |